In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("delhi-weather-aqi-2025.csv")

df["datetime"] = pd.to_datetime(
    df["date_ist"] + " " + df["time_ist"],
    dayfirst=True
)

df = df.sort_values(["location", "datetime"]).reset_index(drop=True)

### Time Features

In [3]:
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)

def season(month):
    if month in [12,1,2]:
        return "winter"
    elif month in [3,4,5]:
        return "summer"
    elif month in [6,7,8,9]:
        return "monsoon"
    else:
        return "post-monsoon"

df["season"] = df["month"].apply(season)

### Lag & Rolling Features

In [4]:
for lag in [1, 3, 6]:
    df[f"aqi_lag_{lag}"] = df.groupby("location")["aqi_index"].shift(lag)

df["aqi_roll_3"] = (
    df.groupby("location")["aqi_index"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

df["pm25_roll_3"] = (
    df.groupby("location")["pm2_5"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

### Interaction Features

In [5]:
df["wind_pm25"] = df["windspeed_kph"] * df["pm2_5"]
df["humidity_pm25"] = df["humidity"] * df["pm2_5"]

### Encoding

In [6]:
df = pd.get_dummies(df, columns=["location", "season"], drop_first=True)

# Drop rows with NaNs from lag features
df = df.dropna().reset_index(drop=True)

df.head()

# Save processed data
df.to_csv("processed_data.csv", index=False)